In [30]:
import time
import numpy as np
import matplotlib.pyplot as plt
import corner
import emcee
from scipy.optimize import differential_evolution

from mond_model import orbit_model_mond
from data_loader import S2DataLoader

## Parameter Names

In [31]:
PARAM_NAMES = ["a", "e", "i", "omega", "Omega", "T0", "M", "a0"]
PARAM_LABELS = [r"$a$ [$10^3$ AU]", r"$e$", r"$i$ [deg]", r"$\omega$ [deg]",
                r"$\Omega$ [deg]", r"$T_0$ [yr]", r"$M$ [$10^6\,M_\odot$]", r"$a_0$"]

## Loading Data

In [41]:
# ==============================================================================
loader = S2DataLoader()
loader.load_data()
data = loader.get_data()
 
t_ast, RA_obs, Dec_obs = data['t_ast'], data['ra_obs'], data['dec_obs']
sigma_RA, sigma_Dec = data['ra_err'], data['dec_err']
t_rv, RV_obs, sigma_RV = data['t_rv'], data['rv_obs'], data['rv_err']
n_ast = len(t_ast)
t_obs = np.concatenate([t_ast, t_rv])
 
print(f"n_astrometric={n_ast}, n_RV={len(t_rv)}")
print(f"t range: [{t_obs.min():.2f}, {t_obs.max():.2f}]")
print(f"RA range: [{RA_obs.min():.4f}, {RA_obs.max():.4f}] arcsec")
print(f"RV range: [{RV_obs.min():.1f}, {RV_obs.max():.1f}] km/s")

n_astrometric=145, n_RV=44
t range: [1992.22, 2016.53]
RA range: [-0.0391, 0.0712] arcsec
RV range: [-1571.0, 1199.0] km/s


## Likelihood and Priors

In [33]:
def log_likelihood(theta):
    try:
        RA_mod, Dec_mod, RV_mod = orbit_model_mond(t_obs, theta, n_ast)
        if len(RA_mod) != len(RA_obs) or len(RV_mod) != len(RV_obs):
            return -np.inf
        chi2 = (np.sum(((RA_mod - RA_obs) / sigma_RA) ** 2) +
                np.sum(((Dec_mod - Dec_obs) / sigma_Dec) ** 2) +
                np.sum(((RV_mod - RV_obs) / sigma_RV) ** 2))
        if not np.isfinite(chi2):
            return -np.inf
        return -0.5 * chi2
    except Exception:
        return -np.inf

BOUNDS = [
    (0.3, 3.0),          # a, 1e3 AU
    (0.01, 0.98),        # e
    (0.0, 180.0),        # i, deg
    (0.0, 360.0),        # omega, deg
    (0.0, 360.0),        # Omega, deg
    (1992, 2016),      # T0, yr - data-driven, not hand-guessed
    (1.0, 8.0),          # M, 1e6 Msun
    (0.001, 0.5),        # a0 - see mond_model.py docstring re: physical units
]


def log_prior(theta):
    for val, (lo, hi) in zip(theta, BOUNDS):
        if not (lo < val < hi):
            return -np.inf
    return 0.0


def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ll = log_likelihood(theta)
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

## Initial Values

In [44]:
# Running MCMC
ndim = 8
nwalkers = 20
nsteps = 5000
burn = 1000
thin = 20
initial = np.array([
    1.03,     # a: ~1026 AU / 1000
    0.884,    # e
    134.2,    # i, deg
    66.1,     # omega, deg
    228.2,    # Omega, deg
    2018.38,  # T0, yr (most recent well-measured pericenter passage)
    4.30,     # M: ~4.30e6 Msun / 1e6
    0.05,     # a0
])

spread = np.array([hi - lo for lo, hi in BOUNDS]) * 1e-3
pos = initial + spread * np.random.randn(nwalkers, ndim)
for j, (lo, hi) in enumerate(BOUNDS):
    pos[:, j] = np.clip(pos[:, j], lo + 1e-6, hi - 1e-6)

In [45]:
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior)
print("Running MCMC...")
sampler.run_mcmc(pos, nsteps, progress=True) # Runs each walker for nsteps iterations.
samples = sampler.get_chain(discard=burn, thin=thin,flat=True) # A 2D array of shape (nsteps * nwalkers, ndim) 
#containing all sampled parameter values.

Running MCMC...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [16:01<00:00,  5.20it/s]


In [67]:
custom = [
    (1.1, 1.2),      # a, 1e3 AU
    (0.9, 0.98),    # e
    (120, 140.0),    # i, deg
    (0.0, 360.0),    # omega, deg
    (0.0, 360.0),    # Omega, deg
    (T0_LO, T0_HI),  # T0, yr
    (1.0, 8.0),      # M, 1e6 Msun
    (0.001, 0.5),    # a0
]
def make_corner_plot(samples, outpath='corner-mond_10k.pdf'):
    fig = corner.corner(
        samples, labels=PARAM_LABELS, quantiles=[0.16, 0.5, 0.84],
        show_titles=True, title_fmt='.3f', range=custom,
        title_kwargs={"fontsize": 10, "pad": 5}, label_kwargs={"fontsize": 10},
        color="#3498db", smooth=1.0, bins=40, plot_datapoints=False,
        fill_contours=True, levels=[0.68, 0.95],
        hist_kwargs={"density": True, "edgecolor": "k", "linewidth": 0.9,
                     "histtype": "stepfilled", "alpha": 0.3},
        contour_kwargs={"linewidths": 1.8, "linestyles": "solid"},
    )
    fig.set_size_inches(12, 12)
    plt.tight_layout(pad=0.5)
    plt.savefig(outpath)
    plt.close(fig)
    print(f"Saved: {outpath}")

NameError: name 'T0_LO' is not defined